# Speculative Decoding

Wiki reference for [Speculative Decoding](https://ml-viz-ruby.vercel.app/wiki/speculative-decoding).

> **Tip:** use *File → Save a copy in Drive* so your edits persist.

**The idea in one sentence.** A small **draft** model proposes $k$ tokens cheaply, the big
**target** model verifies all of them in one forward pass, and a rejection-sampling acceptance
rule guarantees the emitted tokens follow the target's distribution **exactly** — so you get
2–3× faster decoding with provably zero quality change. Here we implement the algorithm from
scratch on toy bigram models, *prove the distribution claim empirically*, reproduce the wiki's
worked trace, and plot the speedup math.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
rng = np.random.default_rng(7)

## From scratch: draft, verify, accept

We use two tiny **bigram language models** over a 4-word vocabulary: row $i$ of a transition
matrix is the next-token distribution after token $i$. The target $P$ is the "real" model; the
draft $Q$ is a cheap, *correlated-but-wrong* approximation (a mix of $P$ and noise).

One speculative cycle: (1) the draft samples $k$ proposals autoregressively, (2) the target
scores every position in a single pass, (3) proposal $x_i$ is accepted with probability
$\min(1, p(x_i)/q(x_i))$; at the first rejection we resample from the normalized residual
$\max(0, p - q)$ and stop; (4) if everything is accepted, a bonus token is sampled from the
target's distribution at position $k{+}1$ for free.

In [ ]:
VOCAB = ['the', 'cat', 'sat', 'mat']
V = len(VOCAB)

P = rng.dirichlet(np.ones(V) * 2, size=V)                 # target bigram model
Q = 0.7 * P + 0.3 * rng.dirichlet(np.ones(V) * 2, size=V) # draft: close to P, but wrong

def accept_or_resample(p_dist, q_dist, proposal, u, rng):
    """The core rule. Returns (token, accepted?)."""
    ratio = min(1.0, p_dist[proposal] / q_dist[proposal])
    if u < ratio:
        return proposal, True
    residual = np.maximum(p_dist - q_dist, 0.0)
    residual /= residual.sum()
    return rng.choice(V, p=residual), False

def speculative_cycle(ctx, k, rng):
    """One draft-verify cycle starting from token `ctx`. Returns emitted tokens."""
    # (1) draft proposes k tokens autoregressively
    proposals, cur = [], ctx
    for _ in range(k):
        cur = rng.choice(V, p=Q[cur])
        proposals.append(cur)
    # (2) target 'verifies': one pass gives p at every position (incl. the bonus position)
    prev = [ctx] + proposals
    # (3) left-to-right accept/reject
    out = []
    for i, tok in enumerate(proposals):
        emitted, ok = accept_or_resample(P[prev[i]], Q[prev[i]], tok, rng.random(), rng)
        out.append(emitted)
        if not ok:
            return out                       # stop at first rejection
    # (4) all accepted -> free bonus token from the target's k+1-th distribution
    out.append(rng.choice(V, p=P[prev[k]]))
    return out

demo = speculative_cycle(0, k=3, rng=np.random.default_rng(0))
print('one cycle from context "the":', [VOCAB[t] for t in demo])

### Validate: the output distribution is *exactly* the target's

The whole point of the acceptance rule. We run 200,000 cycles from a fixed context and compare
the empirical distribution of the **first emitted token** against the target row $P[\text{ctx}]$ —
and, as a control, against the draft row $Q[\text{ctx}]$. If the math is right, the output matches
$P$ to within sampling noise even though every token was *proposed* by $Q$.

In [ ]:
ctx, n_trials = 0, 200_000
counts = np.zeros(V)
for _ in range(n_trials):
    counts[speculative_cycle(ctx, k=3, rng=rng)[0]] += 1
empirical = counts / n_trials

print('token   target p   draft q   speculative output')
for i, w in enumerate(VOCAB):
    print(f'{w:>5}   {P[ctx][i]:8.4f}  {Q[ctx][i]:8.4f}   {empirical[i]:8.4f}')

assert np.abs(empirical - P[ctx]).max() < 0.01, 'output must match the TARGET distribution'
assert np.abs(empirical - Q[ctx]).max() > 0.02, 'output must NOT follow the draft (control)'
print('\nverified: speculative output == target distribution, despite draft proposals')

## Reproduce the worked trace

The wiki's trace: proposals **the → cat → sat** with fixed uniform draws $u = (0.41, -, 0.55)$.
Positions 1–2 accept ($\min(1, 0.72/0.80) = 0.90 > 0.41$; $0.66/0.60 \Rightarrow$ auto-accept);
position 3 rejects ($0.14/0.70 = 0.20 < 0.55$) and resamples from the residual, which puts
92.3% of its mass on **mat** — the token the draft badly under-proposed.

In [ ]:
# position-3 distributions from the wiki table, over (the, cat, sat, mat)
p3 = np.array([0.10, 0.06, 0.14, 0.70])
q3 = np.array([0.05, 0.15, 0.70, 0.10])

assert min(1.0, 0.72 / 0.80) > 0.41, 'position 1 accepts'
assert min(1.0, 0.66 / 0.60) == 1.0, 'position 2 auto-accepts (p >= q)'
assert min(1.0, p3[2] / q3[2]) < 0.55, 'position 3 rejects'

residual = np.maximum(p3 - q3, 0.0)
residual /= residual.sum()
print('residual distribution:', dict(zip(VOCAB, residual.round(4))))
assert np.allclose(residual, [0.0769, 0.0, 0.0, 0.9231], atol=1e-3)
print('trace reproduced: cycle emits (the, cat, mat) — 3 tokens for one target pass')

## Visualize: expected tokens per cycle and wall-clock speedup

With per-token acceptance rate $\alpha$, a cycle emits
$\mathbb{E}[\text{tokens}] = \frac{1-\alpha^{k+1}}{1-\alpha}$, and if a draft step costs a
fraction $c$ of a target step the speedup is that divided by $1 + ck$. We plot both, and overlay
the *simulated* tokens-per-cycle of our toy models to check the formula against reality.

In [ ]:
def expected_tokens(alpha, k):
    return (1 - alpha ** (k + 1)) / (1 - alpha)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ks = np.arange(1, 13)
for alpha, color in [(0.9, '#4ade80'), (0.8, '#facc15'), (0.6, '#f87171')]:
    ax1.plot(ks, [expected_tokens(alpha, k) for k in ks], 'o-', ms=4, color=color,
             label=f'alpha = {alpha}')
    ax2.plot(ks, [expected_tokens(alpha, k) / (1 + 0.05 * k) for k in ks], 'o-', ms=4,
             color=color, label=f'alpha = {alpha}')

# overlay: simulated tokens/cycle for our toy P,Q at several k
sim_rng = np.random.default_rng(3)
sim_k, sim_tok = [2, 4, 8], []
for k in sim_k:
    lens = [len(speculative_cycle(0, k, sim_rng)) for _ in range(20_000)]
    sim_tok.append(np.mean(lens))
ax1.plot(sim_k, sim_tok, 'w*', ms=14, label='simulated (toy P, Q)')

ax1.set(xlabel='lookahead k', ylabel='E[tokens] per target pass', title='Tokens per cycle')
ax2.set(xlabel='lookahead k', ylabel='speedup vs plain decoding',
        title='Wall-clock speedup (draft cost c = 0.05)')
ax1.legend(); ax2.legend()
plt.tight_layout(); plt.show()

**What to notice.**

- **Diminishing returns in $k$ are steep**: $\alpha^k$ decays fast, so the tokens-per-cycle
  curve flattens around $k \approx 4$–8; past that, extra draft steps cost latency ($1 + ck$
  keeps growing) while adding almost no accepted tokens — the speedup curve *turns over*.
- **Acceptance rate is everything**: at $\alpha = 0.9$ the practical speedup approaches 3–4×;
  at $\alpha = 0.6$ it barely clears 1.5×. This is why draft-model choice (or EAGLE-style
  heads trained on the target's own hidden states) matters more than tuning $k$.
- The white stars sit on the theory curve for the matching simulated acceptance rate —
  the geometric model is accurate.

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **draft/target tokenizer mismatch** | proposals can't be verified position-wise; use same tokenizer family |
| **large serving batches** | verification competes with other requests' compute — the win shrinks toward zero |
| **high-temperature generation** | $p$ and $q$ agree less, acceptance drops, cycles waste draft work |
| **`p == q` residual edge case** | residual mass is 0 when the draft equals the target — guard the normalizer |
| **measuring quality "to be safe"** | unnecessary: the emitted distribution is provably identical to the target's (verified above) |

## ✏️ Your turn

**Exercise.** Implement `speedup(alpha, k, c)`: expected wall-clock speedup of speculative
decoding versus plain autoregressive decoding.

Steps: expected tokens per cycle is $(1-\alpha^{k+1})/(1-\alpha)$; one cycle costs $k$ draft
steps at relative cost $c$ plus 1 target step; speedup = expected tokens ÷ cycle cost.

In [ ]:
def speedup(alpha, k, c):
    """
    alpha: per-token acceptance probability (0 < alpha < 1)
    k:     draft lookahead length
    c:     cost of one draft step relative to one target step (e.g. 0.05)
    """
    # TODO(you): expected tokens emitted per cycle
    e_tokens = ...
    # TODO(you): divide by the cycle cost (k draft steps + 1 target step)
    return ...

print(speedup(0.8, 4, 0.05))

In [ ]:
# Assertion — passes silently when your implementation is correct
got = speedup(0.8, 4, 0.05)
assert abs(got - ((1 - 0.8 ** 5) / 0.2) / 1.2) < 1e-9, f'expected ~2.80, got {got}'
assert speedup(0.9, 4, 0.05) > speedup(0.6, 4, 0.05), 'higher acceptance -> more speedup'
assert speedup(0.8, 40, 0.05) < speedup(0.8, 4, 0.05), 'huge k burns draft compute'
assert abs(speedup(0.8, 1, 0.0) - 1.8) < 1e-9, 'free draft, k=1: 1 + alpha tokens per pass'
print('all checks passed')

<details><summary>Solution</summary>

```python
def speedup(alpha, k, c):
    e_tokens = (1 - alpha ** (k + 1)) / (1 - alpha)
    return e_tokens / (1 + c * k)
```

At $\alpha=0.8$, $k=4$, $c=0.05$: $3.36$ tokens per cycle ÷ $1.2$ target-equivalents ≈ **2.8×**.
</details>

## Key takeaways

- **Draft cheap, verify once, accept with $\min(1, p/q)$, resample rejections from
  $\max(0, p-q)$** — between 1 and $k{+}1$ tokens per target pass.
- **The output distribution is exactly the target's (verified empirically)** — a bad draft
  makes it slower, never wrong. Quality evals after enabling it are a nice-to-have, not a gate.
- **$\mathbb{E}[\text{tokens}] = (1-\alpha^{k+1})/(1-\alpha)$** — acceptance rate dominates;
  $k$ beyond ~4–8 turns the speedup curve back down.
- It wins on **templated, low-entropy text at small batch sizes**; it fades at large batches
  where the GPU is already compute-saturated.

**Next:** [the wiki page](https://ml-viz-ruby.vercel.app/wiki/speculative-decoding) ·
[Optimizing LLM Inference](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/22-optimizing-llm-inference)